In [ ]:
import pandas as pd
import re

INPUT_CSV = "tangram_believable_swaps_old.csv"
OUTPUT_CSV = "tangram_believable_swaps.csv"

df = pd.read_csv(INPUT_CSV)

def clean_col(col):
    return df[col].fillna("").astype(str).str.strip()

df["photo_plus_part_description_shape"] = (
    clean_col("photo_description") + " " + clean_col("part_description_shape")
).str.strip()

df["photo_plus_part_description_one_shape_swap"] = (
    clean_col("photo_description") + " " + clean_col("same_picture_1_shape_swap_1")
).str.strip()

df["photo_plus_part_description_shape_loose_swap"] = (
    clean_col("photo_description") + " " + clean_col("loose_swap_item_1")
).str.strip()

df["photo_plus_part_description_second_one_shape_swap"] = (
    clean_col("photo_description") + " " + clean_col("same_picture_1_shape_swap_2")
).str.strip()

def replace_first_part_description(row):
    desc = str(row.get("same_picture_1_shape_swap_1", "")).strip()
    replacement = str(row.get("same_picture_1_shape_swap_1_replacement_part", "")).strip()

    if not desc or not replacement:
        return desc

    # Replace the words between "The" and "of"
    # Example: "The body of the golfer is an orange square."
    # -> "The head of the golfer is an orange square."
    return re.sub(
        r"^The\s+.*?\s+of\s+",
        f"The {replacement} of ",
        desc,
        count=1,
        flags=re.IGNORECASE
    )

df["same_picture_1_shape_swap_1_corrected_part"] = df.apply(
    replace_first_part_description,
    axis=1
)

df["photo_plus_part_description_one_shape_swap_corrected_part"] = (
    clean_col("photo_description")
    + " "
    + df["same_picture_1_shape_swap_1_corrected_part"].fillna("").astype(str).str.strip()
).str.strip()

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved updated CSV to {OUTPUT_CSV}")
print(df.head())